### Explanation of my contribution to the ECHR Freedom of Expression project

First of all we are getting the dataset from the ECHR Open Data website [https://echr-opendata.eu/download/]. We are downloading the Unstructured Cases description + Parsed judgments JSON file.

The raw file needs some processing, mainly:
- Extracting each section in Markdown format (we get the law, facts and conclusion sections)
- Getting the full text from the case (either by fetching the html content from the hudoc page of each case or getting the csv from a local source, after downloading it once).
- Keeping only the relevant columns ('itemid', 'docname','article','appno','judgementdate', 'law', 'facts', 'the_conclusion', 'full_text', 'respondent')


In order to select the relevant ECHR cases (concering Article 10) we use regular expressions to get the cases that are referring to violation or non-violation of Article 10

### Verb Pattern

The verb group 'Holds|Finds|Declares|Rules|Considers|Decides' exhausts all the possible verbs that can be found in the relevant cases. 

In [ ]:
verb_group = r'\b(Holds|Finds|Declares|Rules|Considers|Decides)\b'

This regex pattern matches any of the verbs listed: Holds, Finds, Declares, Rules, Considers, or Decides.
The \b ensures that these words are matched as whole words (not as parts of longer words).

### Violation Pattern

In [ ]:
violation_phrase = r'(violation|violations|breach) of Article 10(?: of the Convention)?'

The (?: of the Convention)? part makes "of the Convention" optional (non-capturing group). It generally captures all the ways a violation can be mentioned in the text

In [ ]:
violation_pattern = rf'(?is){verb_group}.*?\b(has been|have been|is|was|constitutes)\b(?:(?!\bno\b).)*?{violation_phrase}'

This regex pattern is designed to detect sentences that indicate a violation of Article 10.
- (?is): Enables case-insensitive (i) and single-line (s) matching (allowing . to match newlines).
- {verb_group}: Matches one of the verbs (e.g., "Holds", "Finds").
- .*?: Matches any character (non-greedy) between the verb and the next part.
- \b(has been|have been|is|was|constitutes)\b: Matches phrases like "has been", "is", "was", or "constitutes".
- (?:(?!\bno\b).)*?: Ensures that the word "no" does not appear between the verb and the violation phrase (negative lookahead).
- {violation_phrase}: Matches the violation phrase (e.g., "violation of Article 10").


### No Violation Pattern

In [ ]:
no_violation_pattern = rf'(?is){verb_group}.*?\b(has been|is|was|constitutes)\b.*?\bno\b.*?{violation_phrase}'

This regex pattern is designed to detect sentences that indicate no violation of Article 10.
- (?is): Case-insensitive and single-line matching.
- {verb_group}: Matches one of the verbs (e.g., "Holds", "Finds").
- .*?: Matches any character (non-greedy) between the verb and the next part.
- \b(has been|is|was|constitutes)\b: Matches phrases like "has been", "is", "was", or "constitutes".
- .*?\bno\b.*?: Ensures that the word "no" appears between the verb and the violation phrase.
- {violation_phrase}: Matches the violation phrase (e.g., "violation of Article 10").


We search in the text between FOR THESE REASONS and Done in English in the conclusion (using the truncate_text function) for the violation_pattern and no_violation_pattern, matching newlines with the re.DOTALL keyword.

We create 3 classes, "violation_of_article_10", "no_violation_of_article_10" and "other" (for both violation and non violation, as well as no mention of Article 10)

Violation cases length is  496 <br>
Non-Violation cases length is  142 <br>
Everything else cases length is  15458 <br>


### Creating the data validation set

We choose 142 non-violation cases (the entirety of the subset), 142 violation cases and 142 other cases. We shuffle the dataset and create a conclusion text column , a classification column where we store the result of the regex classification and a manual label that is empty, and is left for our validators to fill. We save the file in Excel format.

### Manual Validation

We use [google forms](https://docs.google.com/spreadsheets/d/1TRgkzWr8S5zVygVwOltgjhvwnD9CWF_TmbS2lg0MIPo/edit?gid=911601215#gid=911601215) to upload the excel file created in the previous step. For the manual label we right click, select Data Validation and then Value contains one from list and create the list of values (violation_of_article_10 , non_violation_of_article_10, other)

For the Predictions column we use the condition =IF(D2="", "", IF(C2=D2, "Correct", "Incorrect")) where C2 and D2 are the values in the corresponding Classification and Manual label. 

We create the confusion matrix by again using conditions , like for example in the Actual: violation - predicted violation cell we have =COUNTIFS($D$2:$D$426,"violation_of_article_10",$C$2:$C$426,"violation_of_article_10") where D is the column of manual label (for the actual labels to check if they correspond to violations), and the C column is the predicted labels to see if they also correspond to violations. And so on for the entirety of the matrix.

For the metrics tab, for example for the True Positives violation column of Annotator 1 we simply get the value from the Predicted Violation - Actual Violation from tab Annotator 1.

For the False Negatives of Annotator 1 we get =SUM(Annotator_1!L3:N3) - Annotator_1!L3 , so all the Actual Violations minus the ones where we are correct (so the Actual Violation Predicted Violation column)

For the Kappa Cohen Score of Agreement I am using the sklearn library to calculate it in the calculate_agreement.ipynb, getting the manual labels for annotator 1 and annotator 2, encoding the labels into 0,1,2 and then using the scikit-learn function. 0.99 agreement indicates almost perfect agreement between annotators

| Average Metrics       | violation of Article 10 | no violation of Article 10 | Other         |
|-----------------------|-------------------------|----------------------------|---------------|
| **Precision**         | 1.00                    | 0.98                       | 0.99          |
| **Recall**            | 0.98                    | 1.00                       | 0.99          |
| **F1-Score**          | 0.99                    | 0.99                       | 0.99          |